In [ ]:
import kagglehub
path = kagglehub.dataset_download("msambare/fer2013")

Using Colab cache for faster access to the 'fer2013' dataset.


In [ ]:
!pip install -q timm transformers accelerate


In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

from torchvision import datasets, transforms
from torch.utils.data import Dataset

from transformers import (
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)

import timm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# ======================================================
# PATHS
# ======================================================
train_dir = "/kaggle/input/fer2013/train"
val_dir   = "/kaggle/input/fer2013/train"


In [ ]:
# ==========================================================
# AUGMENTATIONS
# ==========================================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(
        degrees=18,
        translate=(0.08,0.08),
        scale=(0.92,1.08)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3),
    transforms.RandomErasing(
        p=0.30,
        scale=(0.02,0.12)
    )
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])


In [ ]:
# ======================================================
# DATASET
# ======================================================
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir, transform=val_transform)


In [ ]:
# ==========================================================
# IMAGE FOLDER DATASETS
# ==========================================================
train_base = datasets.ImageFolder(train_dir, transform=train_transform)
val_base   = datasets.ImageFolder(val_dir, transform=val_transform)

classes = train_base.classes
num_classes = len(classes)

print("Classes:", classes)
print("Train Samples:", len(train_base))
print("Val Samples:", len(val_base))

print("Classes:", train_base.classes)


Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train Samples: 28709
Val Samples: 28709
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
# ==========================================================
# WRAPPER FOR HF TRAINER
# ==========================================================
class WrapDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img, label = self.ds[idx]
        return {
            "pixel_values": img,
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_ds = WrapDataset(train_base)
val_ds   = WrapDataset(val_base)

In [ ]:
# ==========================================================
# CLASS WEIGHTS (important for FER imbalance)
# ==========================================================
labels = [x[1] for x in train_base.samples]

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

weights = torch.tensor(weights, dtype=torch.float)
print("Class Weights:", weights)


Class Weights: tensor([1.0266, 9.4066, 1.0010, 0.5684, 0.8260, 0.8491, 1.2934])


In [ ]:

# ==========================================================
# MODEL
# ==========================================================
class FaceModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "convnext_tiny",
            pretrained=True,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.embedding = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.40)
        )

        self.classifier = nn.Linear(512, num_classes)

    def forward(self, pixel_values, labels=None):

        feat = self.backbone(pixel_values)
        emb = self.embedding(feat)
        logits = self.classifier(emb)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(
                weight=weights.to(pixel_values.device),
                label_smoothing=0.10
            )
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [ ]:

model = FaceModel()


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [ ]:
# ==========================================================
# METRICS
# ==========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }


In [ ]:
# ==========================================================
# TIME CALLBACK
# ==========================================================
class TimeCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.start = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        t = time.time() - self.start
        print(f"⏱ Epoch Time: {t:.2f} sec")


In [ ]:
# ==========================================================
# TRAINING ARGS
# ==========================================================
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=15,

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    learning_rate=3e-4,
    weight_decay=5e-4,

    lr_scheduler_type="cosine",
    warmup_ratio=0.10,

    save_total_limit=1,

    report_to="none",

    fp16=torch.cuda.is_available(),

    remove_unused_columns=False
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# ==========================================================
# TRAINER
# ==========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[
        TimeCallback(),
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)


In [ ]:
# ==========================================================
# TRAIN
# ==========================================================
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.789624,1.729942,0.515309,0.541335
2,1.668057,1.686347,0.578878,0.559866
3,1.587675,1.624684,0.615765,0.611932
4,1.510649,1.541268,0.666899,0.664685
5,1.446967,1.491501,0.691734,0.683904
6,1.384493,1.442067,0.723850,0.729042
7,1.325124,1.379790,0.753736,0.745288
8,1.264554,1.316408,0.788847,0.787579
9,1.194210,1.222207,0.841478,0.840742
10,1.124103,1.159000,0.873594,0.873507


⏱ Epoch Time: 490.05 sec
⏱ Epoch Time: 282.37 sec
⏱ Epoch Time: 280.68 sec
⏱ Epoch Time: 280.79 sec
⏱ Epoch Time: 283.61 sec
⏱ Epoch Time: 285.47 sec
⏱ Epoch Time: 280.56 sec
⏱ Epoch Time: 281.86 sec
⏱ Epoch Time: 281.14 sec
⏱ Epoch Time: 278.34 sec
⏱ Epoch Time: 276.46 sec
⏱ Epoch Time: 278.85 sec


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.789624,1.729942,0.515309,0.541335
2,1.668057,1.686347,0.578878,0.559866
3,1.587675,1.624684,0.615765,0.611932
4,1.510649,1.541268,0.666899,0.664685
5,1.446967,1.491501,0.691734,0.683904
6,1.384493,1.442067,0.723850,0.729042
7,1.325124,1.379790,0.753736,0.745288
8,1.264554,1.316408,0.788847,0.787579
9,1.194210,1.222207,0.841478,0.840742
10,1.124103,1.159000,0.873594,0.873507


⏱ Epoch Time: 279.24 sec
⏱ Epoch Time: 280.78 sec
⏱ Epoch Time: 285.65 sec


TrainOutput(global_step=6735, training_loss=1.2758290359508397, metrics={'train_runtime': 6526.7922, 'train_samples_per_second': 65.98, 'train_steps_per_second': 1.032, 'total_flos': 0.0, 'train_loss': 1.2758290359508397, 'epoch': 15.0})

In [ ]:
# 1. Save model weights
torch.save(model.state_dict(), "face_model.pt")

print("Model saved successfully!")

Model saved successfully!
